In [1]:
from platform import python_version
print(python_version())

3.11.14


### Calculating DEGs statistics

### For each LFC/FDR cutoff set, we get a different set of DEGs
  - LFC: LFC cutoff and FDR_LFC cutoff
  - Pathway: fdr and pval pathway cutoff and min num of genes

### Up and Down DEGs simulation
  - Up and Down DEGs/DAPs
  - Up and Down in pathways

### there are 2 statistical tables
  - pval/fdr cutoff x degs
  - pval/fdr/geneset/quantile degs_in_pathway, num_pathways

In [ ]:
import os, sys, yaml
from pathlib import Path
from dotenv import load_dotenv

import numpy as np
import pandas as pd
pd.set_option('display.width', 100)
pd.set_option('max_colwidth', 80)
pd.set_option("display.precision", 3)

import seaborn as sns
sns.set_context("notebook", font_scale=1.4)

import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
%matplotlib inline

ROOT0 = Path("/home/flavio/uv/perturb_agent/")
ROOT_SRC = ROOT0 / "src"

sys.path.insert(0, ROOT_SRC)


if str(ROOT_SRC) not in sys.path:
    sys.path.append(str(ROOT_SRC))

print("ROOT0:", ROOT0)
print("ROOT_SRC added:", ROOT_SRC)

from libs.Basic import create_dir
from libs.MTD_lib import MTD
from libs.cBioPortal_lib import cBioPortal
from libs.calc_degs_lib import CALC_DEGS
# from libs.dashcyto_lib import DASH_CYTO
from libs.config_lib import Config
from libs.prism_lib import PRISM


from IPython.display import display, HTML
# display(HTML("<style>.container { width:100% !important; }</style>"))
display(HTML("<style>:root { --jp-notebook-max-width: 100% !important; }</style>"))

with open('../params.yml', 'r') as file:
    dic_yml = yaml.safe_load(file)

# print(dic_yml)

ROOT0: /home/flavio/uv/perturb_agent
ROOT_SRC added: /home/flavio/uv/perturb_agent/src


/home/flavio/uv/perturb_agent/.venv/lib/python3.11/site-packages/Bio/__init__.py:138: BiopythonWarning: You may be importing Biopython from inside the source tree. This is bad practice and might lead to downstream issues. In particular, you might encounter ImportErrors due to missing compiled C extensions. We recommend that you try running your code from outside the source tree. If you are outside the source tree then you have a pyproject.toml file in an unexpected directory: /home/flavio/uv/perturb_agent/.venv/lib/python3.11/site-packages
  warnings.warn(
/home/flavio/uv/perturb_agent/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
email = os.getenv('email')

i_project=0

project_list = dic_yml['project_list']
n = len(project_list)
project = project_list[i_project]

s_project_list = dic_yml['s_project_list']
s_project = s_project_list[i_project]
assert n==len(project_list), f"Error project_list: there are {n} projects"

PROG_ID = 'TCGA'
PSI_ID = 'BRCA'
PSI_ID = 'ACC'
PSI_ID = 'CESC'
PSI_ID = 'PAAD'

ROOT0_DATA = ROOT0 / "data"
root_colab = ROOT0_DATA / 'colab'
root_project = ROOT0_DATA / PROG_ID

disease = PSI_ID

root_project = create_dir(ROOT0_DATA, s_project)
root_disease = create_dir(root_project, PSI_ID)

CONTEXT_DISESE = 'xxxx'
context_disease = CONTEXT_DISESE

gene_protein = dic_yml['gene_protein']
s_omics = dic_yml['s_omics']

has_age = dic_yml['has_age']
has_gender = dic_yml['has_gender']

exp_normalization = dic_yml['exp_normalization']
normalization = 'quantile_norm' if exp_normalization == True else 'not_normalized'

LFC_cut_inf = dic_yml['LFC_cut_inf']
s_pathw_enrichm_method = dic_yml['s_pathw_enrichm_method']
ptw_min_num_of_degs_cut = dic_yml['ptw_min_num_of_degs_cut']

tolerance_pPMI = dic_yml['tolerance_pPMI']
type_sat_ptw_index = dic_yml['type_sat_ptw_index']
saturation_lfc_param = dic_yml['saturation_lfc_param']

pval_pathway_cutoff = dic_yml['pval_pathway_cutoff']
fdr_pathway_cutoff = dic_yml['fdr_pathway_cutoff']
num_of_genes_cutoff = dic_yml['num_of_genes_cutoff']
enr_db_list = dic_yml['enr_db_list']


case_list = dic_yml['case_list']
dic_case_list = dic_yml['dic_case_list']

std_filename      = dic_yml['std_filename']
std_filename_list = dic_yml['std_filename_list']

min_lfc_modulation = dic_yml['min_lfc_modulation']
num_of_genes_list  = dic_yml['num_of_genes_list']
pPMI_normalized  = dic_yml['pPMI_normalized']

#--- max len for formatting purposes
s_len_case  = dic_yml['s_len_case']

n_sentences = dic_yml['n_sentences']
run_list = dic_yml['run_list']
chosen_model_list = dic_yml['chosen_model_list']
i_dfp_list = dic_yml['i_dfp_list']
chosen_model_sampling = dic_yml['chosen_model_sampling']

fdr_ptw_cutoff_list = np.arange(0.05, 0.80, 0.05)
lfc_list = np.round(np.arange(1.0, -0.01, -.025), 3)
fdr_list = np.arange(0.05, 0.76, .01)

cfg = Config(root0=ROOT0, root_disease=root_disease, disease=disease, case_list=case_list)
case = case_list[0]

n_genes_annot_ptw, n_degs, n_degs_in_ptw, n_degs_not_in_ptw, degs_in_all_ratio = -1,-1,-1,-1,-1

LFC_cut, lfc_FDR_cut, n_degs, n_degs_up, n_degs_dw = cfg.get_best_lfc_cutoff(case, 'not_normalized')

print(f"project '{project}', s_project '{s_project}'")
print(f"G/P LFC cutoffs: lfc={LFC_cut:.3f}; fdr={lfc_FDR_cut:.3f} - LFC_cut_inf={LFC_cut_inf:.3f}")
print(f"Pathway cutoffs: pval={pval_pathway_cutoff:.3f}; fdr={fdr_pathway_cutoff:.3f}; num of genes={num_of_genes_cutoff}")

Best parameter file for LFC does not exist /home/flavio/uv/perturb_agent/data/TCGA/PAAD/config/all_lfc_cutoffs_PAAD.tsv
project 'TCGA', s_project 'TCGA'
G/P LFC cutoffs: lfc=1.000; fdr=0.050 - LFC_cut_inf=0.400
Pathway cutoffs: pval=0.050; fdr=0.050; num of genes=3


In [4]:
mtd = MTD(disease=disease, gene_protein=gene_protein, s_omics=s_omics, project=project, s_project=s_project, 
          root0=ROOT0, root0_data=ROOT0_DATA, prog_id=PROG_ID, psi_id=PSI_ID,
          case_list=case_list, dic_case_list=dic_case_list, has_age=has_age, has_gender=has_gender, exp_normalization=exp_normalization,
          std_filename=std_filename, std_filename_list=std_filename_list,
          geneset_num=0, ptw_min_num_of_degs_cut=ptw_min_num_of_degs_cut,
          tolerance_pPMI=tolerance_pPMI, s_pathw_enrichm_method=s_pathw_enrichm_method,
          LFC_cut_inf=LFC_cut_inf, fdr_ptw_cutoff_list=fdr_ptw_cutoff_list,
          num_of_genes_list=num_of_genes_list, lfc_list=lfc_list, fdr_list=fdr_list, 
          min_lfc_modulation=min_lfc_modulation, type_sat_ptw_index=type_sat_ptw_index,
          saturation_lfc_param=saturation_lfc_param, enr_db_list=enr_db_list, pPMI_normalized=pPMI_normalized)

print(">>> Roots", mtd.root0, mtd.root_disease)
case = case_list[0]
print(">>>", mtd.disease, case)

mtd.cfg.set_default_best_lfc_cutoff(mtd.normalization, LFC_cut=1, lfc_FDR_cut=0.05)
ret, degs, degs_ensembl, dfdegs = mtd.open_case(case, prompt_verbose=True, verbose=False)
print("\nEcho Parameters:")
print(mtd.echo_parameters())

>>> Roots /home/flavio/uv/perturb_agent /home/flavio/uv/perturb_agent/data/TCGA/PAAD
>>> PAAD Tumor
>>> case Tumor
>>> psi_id or disease: PAAD
Error: No data available for the specified PAAD.
Error: could not find /home/flavio/uv/perturb_agent/data/TCGA/PAAD/lfc/PAAD_final_LFC_Tumor_x_CTRL_not_normalized.tsv
No dflfc table was calculated for this case Tumor

Echo Parameters:
	0/0 DEGs/ensembl.
		Up 0/0 DEGs/ensembl.
		Dw 0/0 DEGs/ensembl.

Found 0 (best=3) pathways for geneset num=0 'Reactome_Pathways_2024'
Pathway cutoffs p-value=0.050 fdr=0.050 min genes=0.05No enrichment analysis was calculated.


In [5]:
cbio = cBioPortal(root0=ROOT0, root0_data=ROOT0_DATA, memory_restriction=False)

### Get all programs

In [ ]:
verbose = False

df_psi = cbio.open_primary_site(verbose=verbose)
df_psi

### Open primary cites from cbio

In [ ]:
PROG_ID = 'TCGA'
psi_id = 'PAAD'
psi_id = 'SKCM'
psi_id = 'BRCA'

PROG_ID = 'CPTAC2'
psi_id = 'BRCA'

PROG_ID = 'CPTAC3'
psi_id = 'PAAD'

### Prism - development

In [ ]:
import anndata as ad

prism = PRISM(root0=ROOT0, root0_data=ROOT0_DATA)

verbose=True

prism.set_program_and_primary_site(prog_id=PROG_ID, psi_id=psi_id, verbose=verbose)

prism.root_singc, prism.root_singc.exists()

In [ ]:
prism.load_and_view_matrix_txt("count-matrix.txt", nrows=10)

In [ ]:
compression = "gzip"

fname = "count-matrix.txt"
fname_ad = fname.replace('.txt', '.h5ad')

filename_ad = prism.root_singc / fname_ad
# adata.write_h5ad(filename_ad, compression=compression)

print(f"AData saved as {filename_ad},  ({filename_ad.stat().st_size/1e6:.0f} MB), compressed with {compression}")

### Running prism

In [ ]:
fname_celltype="all_celltype.txt"

filename = prism.root_singc / fname_celltype

df_ct = pd.read_csv(filename, sep="\t", index_col=0)
df_ct

In [ ]:
verbose=True

adata = prism.attach_celltypes(adata=adata, fname_celltype="all_celltype.txt", verbose=verbose)

In [ ]:
rep = prism.check_reference(adata=adata)
assert rep["X_looks_like_raw_counts"], rep["problems"]
# adata.write_h5ad("PAAD_Peng2019.h5ad")      # ~500 MB, seconds to reload

In [ ]:
ref, s2t = prism.pseudobulk_reference(adata)

### Calc expression

 - calc_file_expression_tumor_normal_gtex()
   - get_dic_expression_tumor_and_normal()
     - get_filtered_tables()
     - get_table_given_fileID()
   - prepare_normal_tumor_tables()

  
#### Tables in: root_disease / lfc

In [ ]:
cbio.root_disease, cbio.root_lfc

In [ ]:
cbio.root0_data

### Get cases, subtypes and clin_demo tables

In [ ]:
verbose=False
force=False

PROG_ID = 'CPTAC2'
psi_id = 'BRCA'

PROG_ID = 'TCGA'
psi_id = 'SKCM'
psi_id = 'BRCA'
psi_id = 'PAAD'

PROG_ID = 'CPTAC3'
psi_id = 'PAAD'

cbio.set_program_and_primary_site(prog_id=PROG_ID, psi_id=psi_id, verbose=verbose)

df_cases, df_subt, df_clin_demo, df_case_bar = cbio.get_cases_and_subtypes(batch_size=200, force=force, verbose=verbose)

df_cases.shape, df_clin_demo.shape, df_case_bar.shape

In [ ]:
verbose=False

# dic_tumor, dic_normal = cbio.get_dic_expression_tumor_and_normal(verbose=verbose)

In [ ]:
verbose=False
force=False

imax_tumor=200
imax_normal=100

df_tumor, df_normal, df_gtex_ctrl = cbio.calc_file_expression_tumor_normal_gtex(
            imax_tumor=imax_tumor, imax_normal=imax_normal, force=force, verbose=verbose)

print(df_tumor.shape[1], df_normal.shape[1], df_gtex_ctrl.shape[1])

In [ ]:
df_tumor.head(3)

In [ ]:
df_normal.head(3)

In [ ]:
df_gtex_ctrl.head(3)

### Data treatment

1. get raw dfc
2. filter low-expression genes
3. normalize for library size
4. variance-stabilizing transformation
5. select most variable genes
6. cluster samples into k = 3..8 groups
7. evaluate clusters
8. find gene signatures for each cluster

A low-expression gene can be biologically important and even differentially expressed, especially if it is a transcription factor, cytokine, receptor, lncRNA, or rare-cell marker.

But for unsupervised tumor clustering, we usually do not want thousands of genes with mostly zero/very low counts because they add noise and unstable distances.

In [ ]:
verbose=False
force=False

imax_tumor=250
imax_normal=50

exclude_prog_list=['CCLE']
disease_cd = 'PAAD'

dfn_tumor, dfn_normal, df_gtex, df_summ = cbio.get_all_data_from_disease(disease_cd=disease_cd, 
                                                           imax_tumor=imax_tumor, imax_normal=imax_normal,
                                                           exclude_prog_list=exclude_prog_list,
                                                           force=force, verbose=verbose)
print("\n")
print(">> dfn_tumor", dfn_tumor.shape)
print(">> dfn_normal", dfn_normal.shape)

In [ ]:
dfn_tumor.head(3)

In [ ]:
# cbio.plot_boxplot_expression(dfn_tumor, do_log10=True, title = "Expression across tumor samples")

In [ ]:
# cbio.plot_boxplot_expression(dfn_normal, do_log10=True, title = "Expression across normal samples")

### Prism


load_cra001160.py  
=================

Convert the Peng 2019 GSA deposit into an AnnData ready for
`paad_deconv.pseudobulk_reference()`.

Input (from ftp://download.big.ac.cn/gsa/CRA001160/):
- count-matrix.txt   2.77 GB dense TSV, genes x cells
- all_celltype.txt   2.1 MB, per-cell annotation

The matrix is dense text: ~20k genes x ~57k cells is ~1.1e9 values, which is 10-13 GB as a dense float array but well under 1 GB as CSR, since scRNA counts are >90% zeros. So it is parsed in row chunks and sparsified incrementally -- never materialised dense.

```Bash
lftp -e "cls -l; quit" ftp://download.big.ac.cn/gsa/CRA001160/

lftp -e "pget -n 8 -c count-matrix.txt; \
         get all_celltype.txt; get md5sum.txt; quit"      ftp://download.big.ac.cn/gsa/CRA001160/
```


In [ ]:
!python -m ipykernel install --user --name renv --display-name "Python (renv)"

In [ ]:
df_bulk, meta = prism.build_bulk_matrix(dfn_tumor, dfn_normal, cbio.df_metadata, keep_biotypes=("protein_coding", "lncRNA", "miRNA"))

In [ ]:
print(dfn_tumor.shape, dfn_normal.shape, df_bulk.shape)
df_bulk.head(3)

In [ ]:
gene_subset=prism.select_genes(ref)
len(gene_subset)

In [ ]:
res1 = prism.run_bayesprism(df_bulk=df_bulk, ref=ref, state_to_type=s2t, gene_subset=prism.select_genes(ref))
print(len(res1.genes))

In [ ]:
res1.states

In [ ]:
res1.theta_type.columns

### Ductal cell type 1

"Ductal cell type 1" is the normal-like ductal population and stays in the environment compartment — which is what you want. If both had been mapped to malignant, purity would inflate. Verify with res1.tumor_purity.groupby(meta["condition"]).describe(): normals near zero, tumors somewhere in 0.2–0.6.


### Ductal cell type 2

One malignant state means no Ductal cell type 2 subdivision, so subtype_malignant scores Moffitt signatures on a single pooled malignant profile. That still works — it's per-sample expression, so samples can differ — but it won't give you distinct malignant states in θ. For that you'd subcluster Ductal cell type 2 in the AnnData and write finer cell_state labels before calling pseudobulk_reference.

In [ ]:
res1.cell_type_expression("Ductal cell type 1").shape

In [ ]:
res1.cell_type_expression("Ductal cell type 2").shape

### 1. fit theta on markers  (fast, stable)

In [ ]:
res = prism.run_bayesprism(df_bulk=df_bulk, ref=ref, state_to_type=s2t, gene_subset=gene_subset)
type(res)

### 2. theta is now fixed -> expand Z to every gene

In [ ]:
Zfull, gfull = prism.full_Z(res, df_bulk, ref)

In [ ]:
dic = {}

for cell_state in res1.states:
    print(cell_state)
    Z = prism.state_expression(Zfull, gfull, res, cell_state)
    dic[cell_state] = Z

In [ ]:
i=0
key = list(dic.keys())[i]

print(key)
dic[key]

### Ductal 2 - malignant

In [ ]:
Zmal = prism.state_expression(Zfull, gfull, res1, "Ductal cell type 2")

In [ ]:
for g in ["FAM83A-AS1", "HOXA10-AS", "HOXB-AS3", "MIR7-3HG"]:
    if g in gfull:
        print(g, prism.gene_compartment_share(Zfull, gfull, res, g).head(3).round(3).to_dict())

In [ ]:
prog1 = ["FAM83A-AS1", "HOXA10-AS", "HOXB-AS3", "HOXB-AS4", "MIR7-3HG"]

prog2 = ["GATA6", "KRT17", "NEAT1", "H19", "DLEU1", "DLEU2"]

### survived build_bulk_matrix?

> Almost certainly df_bulk is the culprit: build_bulk_matrix defaults to keep_biotypes=("protein_coding",), which removes every lncRNA. Rebuild with them included:

In [ ]:
[g for g in prog1 if g in df_bulk.index]

### present in the scRNA reference?

In [ ]:
  
[g for g in prog1 if g in ref.columns]